# Task 5: LoRA Fine-Tuning

Trains `adapter_baroque` and `adapter_romantic` on cleaned MAESTRO windows.

**Prerequisites**: smoke test passed, MAESTRO filtered (manifest.csv exists).

In [ ]:
# Cell 1: Mount Drive and clone repo
from google.colab import drive
drive.mount('/content/drive')

import subprocess
subprocess.run('git clone https://github.com/YOUR_USERNAME/YOUR_REPO.git /content/thesis', shell=True)
import os; os.chdir('/content/thesis')

In [ ]:
# Cell 2: Install dependencies
exec(open('colab/setup_cell.py').read())

In [ ]:
# Cell 3: Config — edit these paths
import yaml
with open('config.yaml') as f:
    cfg = yaml.safe_load(f)

# Override paths for Colab
cfg['data']['maestro_csv'] = '/content/maestro-v3.0.0/maestro-v3.0.0.csv'
cfg['data']['maestro_midi_dir'] = '/content/maestro-v3.0.0/'
cfg['data']['output_dir'] = '/content/thesis_data'
cfg['training']['drive_dir'] = '/content/drive/MyDrive/thesis_adapters'

with open('config.yaml', 'w') as f:
    yaml.dump(cfg, f)
print('Config updated.')

In [ ]:
# Cell 4: Train Baroque adapter
import subprocess
result = subprocess.run(
    'python train_lora.py --config config.yaml --style baroque',
    shell=True, capture_output=True, text=True
)
print(result.stdout[-3000:])
if result.returncode != 0:
    print('STDERR:', result.stderr[-2000:])

In [ ]:
# Cell 5: Train Romantic adapter
result = subprocess.run(
    'python train_lora.py --config config.yaml --style romantic',
    shell=True, capture_output=True, text=True
)
print(result.stdout[-3000:])
if result.returncode != 0:
    print('STDERR:', result.stderr[-2000:])

In [ ]:
# Cell 6: Plot training curves
import json, matplotlib.pyplot as plt
from pathlib import Path

drive_dir = cfg['training']['drive_dir']
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, style in zip(axes, ('baroque', 'romantic')):
    log_path = Path(drive_dir) / f'adapter_{style}' / 'train_log.jsonl'
    if not log_path.exists():
        ax.set_title(f'{style} — no log')
        continue
    steps, train_losses, val_losses, val_steps = [], [], [], []
    with open(log_path) as f:
        for line in f:
            entry = json.loads(line)
            steps.append(entry['step'])
            train_losses.append(entry['train_loss'])
            if 'val_loss' in entry:
                val_steps.append(entry['step'])
                val_losses.append(entry['val_loss'])
    ax.plot(steps, train_losses, label='train')
    if val_losses:
        ax.plot(val_steps, val_losses, label='val', marker='o')
    ax.set_title(f'{style} adapter')
    ax.set_xlabel('step')
    ax.set_ylabel('loss')
    ax.legend()

plt.tight_layout()
plt.savefig(f'{drive_dir}/training_curves.png', dpi=120)
plt.show()
print('Saved training_curves.png')